# Product review dataset

This notebook uses the English split of the [Multilingual Amazon Reviews Corpus (MARC)](https://huggingface.co/datasets/mteb/amazon_reviews_multi), pinned to dataset revision [`c379a6705fec24a2493fa68e011692605f44e119`](https://huggingface.co/datasets/mteb/amazon_reviews_multi/tree/c379a6705fec24a2493fa68e011692605f44e119). It contains 200,000 training, 5,000 validation, and 5,000 test examples for English. Source files are JSON Lines with `id`, `text`, `label`, and `label_text`; labels are five ordered classes (`0` through `4`).

We download the pinned English splits into `data/raw/amazon_reviews_multi/en/` and loads them as pandas DataFrames. The complete training split is large; run the download cell only when you are ready to fetch it.

In [ ]:
from pathlib import Path
import os
import urllib.request

import pandas as pd

DATASET_REVISION = "c379a6705fec24a2493fa68e011692605f44e119"
DATASET_URL = (
    "https://huggingface.co/datasets/mteb/amazon_reviews_multi/resolve/"
    f"{DATASET_REVISION}/en"
)
SPLITS = ("train", "validation", "test")

working_dir = Path.cwd()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or its notebooks directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "amazon_reviews_multi" / "en"

def download_split(split: str) -> Path:
    """Download one split from the pinned dataset revision, reusing a cached file."""
    if split not in SPLITS:
        raise ValueError(f"Unsupported split {split!r}; choose from {SPLITS}.")

    destination = RAW_DATA_DIR / f"{split}.jsonl"
    if destination.is_file() and destination.stat().st_size > 0:
        return destination

    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(".jsonl.part")
    request = urllib.request.Request(
        f"{DATASET_URL}/{split}.jsonl",
        headers={"User-Agent": "product-review-sentiment-data-loader"},
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            with temporary_path.open("wb") as output:
                while chunk := response.read(1024 * 1024):
                    output.write(chunk)
        if temporary_path.stat().st_size == 0:
            raise RuntimeError(f"The downloaded {split} split is empty.")
        os.replace(temporary_path, destination)
    finally:
        temporary_path.unlink(missing_ok=True)
    return destination

review_splits = {
    split: pd.read_json(download_split(split), lines=True)
    for split in SPLITS
}

required_columns = {"id", "text", "label", "label_text"}
for split, reviews in review_splits.items():
    missing_columns = required_columns - set(reviews.columns)
    if missing_columns:
        raise ValueError(f"{split} is missing expected columns: {sorted(missing_columns)}")
    if reviews.empty or reviews["text"].isna().any() or reviews["text"].str.strip().eq("").any():
        raise ValueError(f"{split} contains no rows or reviews with missing/blank text.")
    if not reviews["label"].isin(range(5)).all():
        raise ValueError(f"{split} contains labels outside the expected 0-4 range.")
    print(f"{split}: {len(reviews):,} reviews; label counts: {reviews['label'].value_counts().sort_index().to_dict()}")

train_reviews = review_splits["train"]
validation_reviews = review_splits["validation"]
test_reviews = review_splits["test"]
train_reviews.head()